# CatBoost: спотовый BTC, прогноз на 15 минут каждые 15 минут

Самостоятельная версия лучшей из проверенных моделей по MAE: **27 исходных + 5 ценовых признаков, без стакана**.
Все признаки и обучение находятся в этом ноутбуке; старые эксперименты не нужны.
Исходник — минутный CSV парсера Binance. Сырые данные не изменяются.

Обучение: ноябрь 2021 — октябрь 2025. Тест: ноябрь 2025 — октябрь 2026.
CatBoost: 200 деревьев, глубина 5, MAE, learning_rate=0.03, L2=10, seed=42.
Результаты ретроспективные: этот тест уже использовался при выборе модели.
Снижение MAE не является доходностью торговой стратегии; по MSE модель может уступать нулю.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

PROJECT = Path("C:/Users/aweso/Desktop/project")
CSV = PROJECT / "kode/crypto-data-mvp/data/history_5y/BTCUSDT_dataset_1m.csv"
OUTPUT = PROJECT / "ml_tasks" / "results"
OUTPUT.mkdir(parents=True, exist_ok=True)

TRAIN_START = pd.Timestamp("2021-11-01", tz="UTC")
TEST_START = pd.Timestamp("2025-11-01", tz="UTC")
TEST_END = pd.Timestamp("2026-11-01", tz="UTC")
GAP = pd.Timedelta(hours=26)  # Сохранён зазор предыдущего сравнения горизонтов.


## 1. Минутные свечи

`timestamp` — начало минуты. Пропущенные минуты сохраняются в сетке как NaN: цены не интерполируются.
Время закрытия минуты на минуту позже её timestamp.

In [2]:
minutes = pd.read_csv(CSV, usecols=["timestamp", "open", "high", "low", "close", "volume_base"])
minutes.index = pd.to_datetime(minutes.pop("timestamp"), unit="ms", utc=True)
assert minutes.index.is_unique and minutes.index.is_monotonic_increasing
minutes = minutes.asfreq("min")

minute_close = minutes["close"].copy()
minute_close.index = minute_close.index + pd.Timedelta(minutes=1)
print(f"Минут в сетке: {len(minutes):,}; без цены: {minutes['close'].isna().sum():,}")


Минут в сетке: 2,737,440; без цены: 200


## 2. Причинный фильтр Калмана

Фильтр проходит цены только вперёд. Параметр Q/R=0.1 фиксирован.
Для отметок :00/:15/:30/:45 используются отдельные часовые сетки, поэтому фильтр не ускоряется вчетверо.

In [3]:
def kalman_log(prices):
    state = None
    variance = 1.0
    result = []
    for price in prices:
        variance += 0.1
        if not np.isfinite(price):
            result.append(np.nan)
            continue
        observation = np.log(price)
        if state is None:
            state = observation
        gain = variance / (variance + 1.0)
        state += gain * (observation - state)
        variance *= 1.0 - gain
        result.append(state)
    return pd.Series(result, index=prices.index)


## 3. Те же 32 признака

Первые 27: прошлые доходности, волатильность часовых доходностей, отклонения от SMA,
относительные объёмы и форма последней часовой свечи.

Пять добавлений: положение цены в диапазоне 24 часов, z-score цены за 168 часов,
остаток Калмана, часовой наклон Калмана и остаток, нормированный на прошлую волатильность.
Известный дубль наклона/остатка сохранён для точного воспроизведения модели.

In [4]:
def make_features(hourly):
    close = hourly["close"]
    returns = close.pct_change(fill_method=None)
    features = pd.DataFrame(index=hourly.index)

    for hours in [1, 4, 12, 24]:
        features[f"return_{hours}h"] = close.pct_change(hours, fill_method=None)
    for hours in [4, 12, 24]:
        features[f"volatility_{hours}h"] = returns.rolling(hours).std()
        features[f"ma_gap_{hours}h"] = close / close.rolling(hours).mean() - 1
    for hours in [4, 24]:
        past_volume = hourly["volume_base"].shift(1).rolling(hours).mean()
        features[f"volume_ratio_{hours}h"] = hourly["volume_base"] / past_volume.replace(0, np.nan)

    features["range"] = (hourly["high"] - hourly["low"]) / close
    features["body"] = close / hourly["open"] - 1
    features["close_position"] = ((close - hourly["low"]) / (hourly["high"] - hourly["low"]).replace(0, np.nan)).fillna(0.5)

    for hours in [72, 168, 336]:
        features[f"return_{hours}h"] = close.pct_change(hours, fill_method=None)
        features[f"volatility_{hours}h"] = returns.rolling(hours).std()
        features[f"ma_gap_{hours}h"] = close / close.rolling(hours).mean() - 1
        past_volume = hourly["volume_base"].shift(1).rolling(hours).mean()
        features[f"volume_ratio_{hours}h"] = hourly["volume_base"] / past_volume.replace(0, np.nan)

    low = hourly["low"].rolling(24).min()
    high = hourly["high"].rolling(24).max()
    features["position_in_range_24h"] = (close - low) / (high - low).replace(0, np.nan)
    features["price_zscore_168h"] = (close - close.rolling(168).mean()) / close.rolling(168).std().replace(0, np.nan)
    level = kalman_log(close)
    residual = np.log(close) - level
    features["kalman_gap_qr_0p1"] = residual
    features["kalman_slope_1h_qr_0p1"] = level.diff()
    features["kalman_gap_vol_scaled_qr_0p1"] = residual / np.log(close).diff().rolling(24).std().replace(0, np.nan)
    return features.replace([np.inf, -np.inf], np.nan)


## 4. Скользящий шаг 15 минут

В 12:15 последняя часовая свеча — 11:15–12:15. Периоды индикаторов остаются в часах:
например, волатильность за 4 часа использует четыре часовые доходности.
Неполные часовые свечи не используются.

Таргет — `100 × ln(close[t+15 минут] / close[t])`.
Для точного воспроизведения прежней выборки сохранён технический фильтр доступности будущих цен
на горизонтах из предыдущего эксперимента. Эти цены участвуют только в проверке наличия таргетов,
никогда не подаются в признаки. Единственная обучаемая цель — 15 минут.

In [5]:
frames = []
check_horizons = [15, 30, 60, 120, 240, 360, 480, 720, 1440]

for phase in [0, 15, 30, 45]:
    options = dict(origin="start_day", offset=pd.Timedelta(minutes=phase), label="right", closed="left")
    hourly = minutes.resample("h", **options).agg({
        "open": "first", "high": "max", "low": "min", "close": "last", "volume_base": "sum"
    })
    count = minutes["close"].resample("h", **options).count()
    hourly = hourly.where(count.eq(60))
    features = make_features(hourly).where(count.eq(60))
    feature_columns = features.columns.tolist()
    assert len(feature_columns) == 32

    valid = features.iloc[:, :27].notna().all(axis=1)
    for horizon in check_horizons:
        future = minute_close.reindex(features.index + pd.Timedelta(minutes=horizon))
        valid &= future.notna().to_numpy()

    future_close = minute_close.reindex(features.index + pd.Timedelta(minutes=15)).to_numpy()
    target = 100 * np.log(future_close / hourly["close"])
    frame = features.loc[valid].copy()
    frame["target_15m"] = target.loc[valid]
    frames.append(frame)

dataset = pd.concat(frames).sort_index()
assert dataset.index.is_unique


## 5. Четыре года обучения и год теста

Все настройки фиксированы. Тест не используется для ранней остановки или подбора параметров.
На годовом тесте модель не переобучается. Ожидаемые размеры прежнего набора:
138 761 обучающее наблюдение и 34 936 тестовых.

In [6]:
train = dataset.loc[(dataset.index >= TRAIN_START) & (dataset.index + GAP < TEST_START)].copy()
test = dataset.loc[(dataset.index >= TEST_START) & (dataset.index + GAP < TEST_END)].copy()
assert train.index.max() + pd.Timedelta(minutes=15) < test.index.min()
assert (test.index + pd.Timedelta(minutes=15) < TEST_END).all()

X_train = train[feature_columns].astype(np.float32)
X_test = test[feature_columns].astype(np.float32)
y_train = train["target_15m"]
y_test = test["target_15m"]
print(f"Обучение: {len(train):,}; тест: {len(test):,}; признаков: {len(feature_columns)}")


Обучение: 138,761; тест: 34,936; признаков: 32


In [7]:
model = CatBoostRegressor(
    iterations=200, depth=5, learning_rate=0.03, l2_leaf_reg=10,
    loss_function="MAE", random_seed=42, thread_count=4,
    task_type="CPU", verbose=False, allow_writing_files=False,
)
model.fit(X_train, y_train)
prediction = model.predict(X_test)


## 6. Метрики с 95% доверительными интервалами

Нулевой прогноз означает нулевую доходность. Положительное снижение ошибки — лучше нуля.
MAE и MSE показываются вместе: преимущество по MAE не гарантирует преимущество по MSE.
Pearson измеряет линейную связь прогноза с фактом; точность направления — отдельная метрика.

Интервалы: парный круговой блочный bootstrap, 2000 повторений, блок 672 последовательных
15-минутных наблюдения (неделя). Для нормированных разниц ошибок знаменатель нулевого
прогноза фиксирован по тесту, как в прежнем расчёте MAE. Поправки на историю подбора нет.

In [8]:
def metric_values(actual, forecast):
    error = actual - forecast
    return np.array([
        np.abs(error).mean(),
        np.abs(actual).mean(),
        (np.abs(actual) - np.abs(error)).mean(),
        (actual ** 2 - error ** 2).mean(),
        np.corrcoef(actual, forecast)[0, 1],
        100 * np.mean((actual > 0) == (forecast > 0)),
    ])


actual = y_test.to_numpy()
values = metric_values(actual, prediction)
rng = np.random.default_rng(42)
block_size = 7 * 24 * 4
block_count = int(np.ceil(len(actual) / block_size))
samples = []

for _ in range(2000):
    starts = rng.integers(0, len(actual), size=block_count)
    rows = (starts[:, None] + np.arange(block_size)) % len(actual)
    rows = rows.ravel()[:len(actual)]
    samples.append(metric_values(actual[rows], prediction[rows]))

samples = np.array(samples)
scale = np.array([1, 1, 100 / values[1], 100 / np.mean(actual ** 2), 1, 1])
values *= scale
samples *= scale
low, high = np.quantile(samples, [0.025, 0.975], axis=0)

metrics = pd.DataFrame({"value": values, "ci_95_low": low, "ci_95_high": high}, index=[
    "MAE модели (log return × 100)",
    "MAE нулевого прогноза",
    "Снижение MAE к нулю, %",
    "Снижение MSE к нулю, %",
    "Pearson",
    "Точность направления, %",
])
display(metrics.round(6))


,value,ci_95_low,ci_95_high
MAE модели (log return × 100),0.150875,0.137979,0.165769
MAE нулевого прогноза,0.151493,0.138537,0.166485
"Снижение MAE к нулю, %",0.407900,0.250943,0.575958
"Снижение MSE к нулю, %",-0.569256,-0.767169,-0.376003
Pearson,0.018460,0.009060,0.028340
"Точность направления, %",52.951111,52.415775,53.486447


## 7. Сохранение

В `ml_tasks/results` сохраняются модель, CSV признаков с таргетом и split,
тестовые прогнозы, метрики с интервалами и настройки. Время, таргет и split не являются входами модели.

In [9]:
model.save_model(str(OUTPUT / "btc_catboost_15m.cbm"))
metrics.to_csv(OUTPUT / "metrics.csv", index_label="metric")
predictions = pd.DataFrame({"actual": y_test, "prediction": prediction})
predictions.to_csv(OUTPUT / "predictions.csv", index_label="datetime_utc")

saved_dataset = pd.concat([train.assign(split="train"), test.assign(split="test")])
saved_dataset.to_csv(OUTPUT / "dataset_15m.csv", index_label="datetime_utc")
settings = {
    "source_csv": str(CSV), "features": feature_columns, "parameters": model.get_params(),
    "train_start": str(TRAIN_START), "test_start": str(TEST_START), "test_end": str(TEST_END),
    "horizon_minutes": 15, "step_minutes": 15, "gap_hours": 26,
    "train_rows": len(train), "test_rows": len(test),
    "bootstrap_repeats": 2000, "bootstrap_block_rows": block_size,
}
(OUTPUT / "settings.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8")
print("Готово: модель и CSV сохранены в ml_tasks/results.")


Готово: модель и CSV сохранены в ml_tasks/results.
